<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9a_JANA_TwoMoons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9a — exact JANA Two Moons and topology-matched hybrid CE

This notebook answers a deliberately narrow question: **if JANA is trained on the exact main-paper Two Moons problem with the exact published BayesFlow architecture and training campaign, what changes when we retain the same flow topology but add our CE correction?**

There are two important controls.

1. Figure 4 of the JANA paper uses the **Wiqvist et al. simulator**, whose moons have radius \(1.0\) and radial noise \(0.1\). `Exercise_9b_SBIBM_TwoMoons.ipynb` uses the Lueckmann/SBIBM simulator, whose moons have radius \(0.1\) and radial noise \(0.01\). The JANA paper treats that second simulator only in its appendix. They are related benchmarks, but they are not the same numerical problem.
2. The published comparison uses legacy BayesFlow, not our 10-layer `nflows` RQS. This notebook runs the historical JANA environment in an isolated Python 3.11 process, leaving the current Colab kernel untouched for the PyTorch classifier.

The experiment produces three frozen density systems:

- **Original JANA:** posterior and likelihood flows trained jointly with the summed NLL, exactly as in the paper code.
- **Matched separate flows:** the same posterior and likelihood topologies, data, epochs, batch size, optimizer, and learning-rate schedule, but optimized through separate trainer instances.
- **Hybrid + CE:** the matched separate flows multiplied by the equal-prior \(S/P\) and \(S/L\) corrections from the same plain CE classifier architecture used in `Exercise_9b_SBIBM_TwoMoons.ipynb`.

No normalization or bridge term enters classifier training. Those identities are evaluated only after all CE checkpoints are frozen.


In [ ]:
# Campaign controls
PROFILE = "PAPER"       # PAPER | SMOKE
SEED = 1                # repetition #1 shown in the JANA main paper
SIMULATION_BUDGET = 10_000 if PROFILE == "PAPER" else 512
FORCE_LEGACY_FLOWS = False
LOAD_CLASSIFIERS_IF_AVAILABLE = True

if PROFILE == "PAPER":
    POSTERIOR_PROPOSALS = 150_000
    PRIOR_PROPOSALS = 150_000
    POSTERIOR_SAMPLES = 10_000
    ANALYTIC_SAMPLES = 10_000
    CLASSIFIER_MEMBERS = 10
    CLASSIFIER_EPOCHS = 250
    CLASSIFIER_WIDTH = 1024
    CLASSIFIER_LAYERS = 4
else:
    POSTERIOR_PROPOSALS = 4_000
    PRIOR_PROPOSALS = 4_000
    POSTERIOR_SAMPLES = 2_000
    ANALYTIC_SAMPLES = 2_000
    CLASSIFIER_MEMBERS = 1
    CLASSIFIER_EPOCHS = 3
    CLASSIFIER_WIDTH = 128
    CLASSIFIER_LAYERS = 2

if PROFILE == "PAPER" and SIMULATION_BUDGET != 10_000:
    raise ValueError("The paper campaign requires exactly 10,000 simulations.")


In [ ]:
# Colab setup: repository, persistent artifacts, and isolated JANA environment.
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
IN_COLAB = "google.colab" in sys.modules

def run(*args, env=None):
    command = [str(arg) for arg in args]
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env)

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    REPO_DIR = Path("/content/nsbi-lhc-toolkit")
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    ARTIFACT_ROOT = Path(
        f"/content/drive/MyDrive/hybrid_nsbi_ml/exercise_9a_JANA_TwoMoons/seed_{SEED:02d}"
    )
    LEGACY_ENV = Path("/content/jana-paper-py311")
else:
    candidates = [
        Path.cwd(),
        Path.cwd() / "workshops" / "ml4hep_tifr_colab",
    ]
    TUTORIAL_DIR = next(
        path for path in candidates
        if (path / "jana_twomoons_paper_driver.py").is_file()
    )
    ARTIFACT_ROOT = (
        Path.cwd() / "exercise_9a_JANA_TwoMoons_artifacts" / f"seed_{SEED:02d}"
    )
    LEGACY_ENV = Path("/tmp/jana-paper-py311")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVER = TUTORIAL_DIR / "jana_twomoons_paper_driver.py"
REQUIREMENTS = TUTORIAL_DIR / "requirements_jana_twomoons_paper.txt"
for required in (DRIVER, REQUIREMENTS):
    if not required.is_file():
        raise FileNotFoundError(required)

if shutil.which("uv") is None:
    run(sys.executable, "-m", "pip", "install", "-q", "uv")
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
UV_ENV = os.environ.copy()
UV_ENV.setdefault("UV_CACHE_DIR", str(LEGACY_ENV.parent / "jana-uv-cache"))
UV_ENV.setdefault("UV_PYTHON_INSTALL_DIR", str(LEGACY_ENV.parent / "jana-uv-python"))
LEGACY_PYTHON = LEGACY_ENV / "bin" / "python"
if not LEGACY_PYTHON.is_file():
    run(UV, "venv", "--python", "3.11", LEGACY_ENV, env=UV_ENV)

requirements_digest = hashlib.sha256(REQUIREMENTS.read_bytes()).hexdigest()
stamp = LEGACY_ENV / ".jana_requirements_sha256"
if not stamp.is_file() or stamp.read_text().strip() != requirements_digest:
    run(
        UV, "pip", "install", "--python", LEGACY_PYTHON,
        "--requirement", REQUIREMENTS,
        env=UV_ENV,
    )
    # JAX is not imported by this experiment. The 2023 requirements did not
    # pin jaxlib, so a 2026 resolver pairs jax 0.4.10 with an incompatible
    # modern jaxlib. Removing both restores TensorFlow import without changing
    # any JANA model, loss, optimizer, simulator, or numerical dependency.
    run(
        UV, "pip", "uninstall", "--python", LEGACY_PYTHON,
        "jax", "jaxlib", env=UV_ENV,
    )
    stamp.write_text(requirements_digest)

if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))
print("Tutorial directory:", TUTORIAL_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)
print("Legacy JANA Python:", LEGACY_PYTHON)


## 1. Exact architecture and optimization contract

The following settings are not inferred from the paper's plots. They are read from `experiments/two_moons/jana.py` at JANA-Paper commit `6cbbc94...` and from BayesFlow commit `153dfef...`.

| Component | Published setting used here |
|---|---|
| Prior | \(\theta\sim\mathcal U([-2,2]^2)\) |
| Observation | \(x_{\rm obs}=(0,0)\) |
| Simulator | Wiqvist main-paper Two Moons |
| Posterior flow | 4 spline coupling layers |
| Likelihood flow | 5 interleaved layers: affine/spline/affine/spline/affine |
| Permutation | Learned orthogonal transform |
| Spline subnet regularization | L2 coefficient \(5\times10^{-3}\), dropout 0.05 |
| Affine subnet regularization | L2 coefficient \(5\times10^{-4}\), dropout 0.01 |
| Within-coupling subnet | 2 dense layers × 128 ReLU units |
| Spline | 16 bins over \([-5,5]\) |
| ActNorm | Enabled |
| Training | 64 epochs, batch size 32 |
| Optimizer | Adam, initial LR \(5\times10^{-4}\), cosine decay to zero |
| Gradient clipping | Global norm 1 |
| Validation | 300 extra monitoring simulations; no early stopping |
| Main budget | 10,000 training simulations |

The original script does not seed TensorFlow. This reproduction adds a TensorFlow seed so that a saved run can be reused; it does not alter the model, loss, optimizer, or data distribution. The original requirements also left `jaxlib` unpinned. Since JAX is unused here and modern resolvers select a `jaxlib` incompatible with the paper's `jax==0.4.10`, the setup removes both JAX packages after installing every published pin. TensorFlow, TensorFlow Probability, BayesFlow, NumPy, and all training-relevant versions remain exactly pinned.


In [ ]:
# Train or reuse the exact legacy flows.
command = [
    str(LEGACY_PYTHON),
    str(DRIVER),
    "--artifact-root", str(ARTIFACT_ROOT),
    "--simulation-budget", str(SIMULATION_BUDGET),
    "--seed", str(SEED),
    "--posterior-proposals", str(POSTERIOR_PROPOSALS),
    "--prior-proposals", str(PRIOR_PROPOSALS),
    "--posterior-samples", str(POSTERIOR_SAMPLES),
    "--analytic-samples", str(ANALYTIC_SAMPLES),
]
if FORCE_LEGACY_FLOWS:
    command.append("--force")
run(*command)


In [ ]:
# Load and validate immutable outputs from the pinned environment.
import numpy as np
import pandas as pd
from IPython.display import display

metadata_path = ARTIFACT_ROOT / "legacy_flow_metadata.json"
arrays_path = ARTIFACT_ROOT / "legacy_flow_outputs.npz"
metadata = json.loads(metadata_path.read_text())
legacy = dict(np.load(arrays_path, allow_pickle=False))

expected = {
    "seed": SEED,
    "simulation_budget": SIMULATION_BUDGET,
    "posterior_proposals": POSTERIOR_PROPOSALS,
    "prior_proposals": PRIOR_PROPOSALS,
    "posterior_samples": POSTERIOR_SAMPLES,
    "jana_paper_commit": "6cbbc94faf0aa85147986f7f9516d13a52551bd4",
    "bayesflow_commit": "153dfefadd347717b7aeb9c4872a4b51ac04e83c",
    "simulator": "Wiqvist_et_al_main_paper_two_moons",
}
mismatches = {
    key: (metadata.get(key), value)
    for key, value in expected.items()
    if metadata.get(key) != value
}
if mismatches:
    raise RuntimeError(f"Incompatible legacy artifacts: {mismatches}")
for name, values in legacy.items():
    if not np.isfinite(values).all():
        raise FloatingPointError(f"Non-finite legacy array {name}")
print(json.dumps({key: metadata[key] for key in expected}, indent=2))
display(pd.DataFrame(
    [{"array": name, "shape": str(values.shape), "dtype": str(values.dtype)}
     for name, values in legacy.items()]
))


## 2. Equal-prior CE correction

Let \(S\), \(P\), and \(L\) denote the three joint laws

\[
S(\theta,x)=p(\theta,x),\qquad
P(\theta,x)=p(x)\,q_\phi(\theta\mid x),\qquad
L(\theta,x)=p(\theta)\,q_\eta(x\mid\theta).
\]

The simulator rows used to train the flows form \(S\). We obtain one frozen-flow draw per matching context for \(P\) and \(L\), preserving triplets during the classifier split. A plain three-class network is trained only with

\[
\mathcal L_{\rm CE}=-\mathbb E\log D_Y(\theta,x).
\]

With equal class priors,

\[
r_P(\theta,x)=\frac{S}{P}=\frac{D_S}{D_P},\qquad
r_L(\theta,x)=\frac{S}{L}=\frac{D_S}{D_L}.
\]

As in the current Exercise 9b, classifier inputs receive one fixed training-column mean/standard-deviation change of units. There is no learned normalization layer, dropout, weight decay, residual block, density feature, normalization loss, or bridge loss.


In [ ]:
# Construct complete S/P/L triplets and one leakage-free classifier split.
theta = legacy["theta_train"].astype(np.float32)
x = legacy["x_train"].astype(np.float32)
theta_p = legacy["theta_p"].astype(np.float32)
x_l = legacy["x_l"].astype(np.float32)

S = np.column_stack([theta, x])
P = np.column_stack([theta_p, x])
L = np.column_stack([theta, x_l])
groups = np.stack([S, P, L], axis=1).astype(np.float32)
if groups.shape != (SIMULATION_BUDGET, 3, 4):
    raise RuntimeError(f"Unexpected grouped classifier shape {groups.shape}")

split_rng = np.random.default_rng(SEED + 10_001)
permutation = split_rng.permutation(len(groups))
n_train = int(0.8 * len(groups))
train_index, validation_index = permutation[:n_train], permutation[n_train:]
if np.intersect1d(train_index, validation_index).size:
    raise RuntimeError("Classifier train/validation leakage")
train_groups_raw = groups[train_index]
validation_groups_raw = groups[validation_index]

flat_train = train_groups_raw.reshape(-1, train_groups_raw.shape[-1])
classifier_center = flat_train.mean(axis=0, dtype=np.float64).astype(np.float32)
classifier_scale = flat_train.std(axis=0, dtype=np.float64).astype(np.float32)
classifier_scale = np.where(classifier_scale > 1e-6, classifier_scale, 1.0).astype(np.float32)

def transform_classifier_points(values):
    values = np.asarray(values, dtype=np.float32)
    return ((values - classifier_center) / classifier_scale).astype(np.float32)

train_groups = transform_classifier_points(train_groups_raw)
validation_groups = transform_classifier_points(validation_groups_raw)
print("Classifier groups:", len(train_groups), "train +", len(validation_groups), "validation")
print("Fixed center:", classifier_center)
print("Fixed scale:", classifier_scale)


In [ ]:
# Train or load the exact Exercise-9b CE-only classifier topology.
import copy
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from utils_exercise9b_contract import (
    INITIAL_LEARNING_RATE,
    LEARNING_RATE_DROP_FACTOR,
    LEARNING_RATE_STEP_EPOCHS,
    MINIMUM_LEARNING_RATE,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_S, CLASS_P, CLASS_L = 0, 1, 2
CLASSIFIER_ROW_BATCH_BUDGET = 32
classifier_checkpoint_dir = ARTIFACT_ROOT / "plain_three_class_classifier"
classifier_checkpoint_dir.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

class PlainThreeClassMLP(nn.Module):
    def __init__(self, input_dim, width, hidden_layers):
        super().__init__()
        layers = []
        n_input = int(input_dim)
        for _ in range(int(hidden_layers)):
            layers.extend([nn.Linear(n_input, int(width)), nn.ReLU()])
            n_input = int(width)
        layers.append(nn.Linear(n_input, 3))
        self.network = nn.Sequential(*layers)

    def forward(self, values):
        return self.network(values)

def scheduled_learning_rate(epoch):
    return max(
        MINIMUM_LEARNING_RATE,
        INITIAL_LEARNING_RATE
        * LEARNING_RATE_DROP_FACTOR ** (int(epoch) // LEARNING_RATE_STEP_EPOCHS),
    )

def set_learning_rate(optimizer, epoch):
    value = scheduled_learning_rate(epoch)
    for group in optimizer.param_groups:
        group["lr"] = value
    return value

def class_ce(model, group_batch):
    group_batch = group_batch.to(device)
    logits = model(group_batch.reshape(-1, group_batch.shape[-1]))
    labels = torch.arange(3, device=device).repeat(len(group_batch))
    return F.cross_entropy(logits, labels)

@torch.no_grad()
def validation_ce(model, values, chunk_size=512):
    model.eval()
    total = 0.0
    for start in range(0, len(values), int(chunk_size)):
        batch = values[start:start + int(chunk_size)]
        total += float(class_ce(model, batch).cpu()) * len(batch)
    return total / len(values)

classifier_config = {
    "input_dim": 4,
    "width": CLASSIFIER_WIDTH,
    "hidden_layers": CLASSIFIER_LAYERS,
    "members": CLASSIFIER_MEMBERS,
    "epochs": CLASSIFIER_EPOCHS,
    "batch_size_row_budget": CLASSIFIER_ROW_BATCH_BUDGET,
    "initial_learning_rate": INITIAL_LEARNING_RATE,
    "minimum_learning_rate": MINIMUM_LEARNING_RATE,
    "learning_rate_drop_factor": LEARNING_RATE_DROP_FACTOR,
    "learning_rate_step_epochs": LEARNING_RATE_STEP_EPOCHS,
    "objective": "equal_prior_multiclass_ce_only",
    "input_transform": "fixed_training_column_mean_std_v1",
    "regularization": "none",
    "seed": SEED,
    "simulation_budget": SIMULATION_BUDGET,
    "flow_artifact_metadata": hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
}

def checkpoint_path(member):
    return classifier_checkpoint_dir / f"member_{member:02d}.pt"

def load_classifier_ensemble():
    if not LOAD_CLASSIFIERS_IF_AVAILABLE:
        return None
    if not all(checkpoint_path(member).is_file() for member in range(CLASSIFIER_MEMBERS)):
        return None
    loaded = []
    for member in range(CLASSIFIER_MEMBERS):
        try:
            saved = torch.load(
                checkpoint_path(member), map_location=device, weights_only=False
            )
        except TypeError:
            saved = torch.load(checkpoint_path(member), map_location=device)
        if saved.get("config") != {**classifier_config, "member": member}:
            raise RuntimeError(f"Incompatible classifier checkpoint {checkpoint_path(member)}")
        if not np.array_equal(np.asarray(saved["center"]), classifier_center):
            raise RuntimeError("Classifier center mismatch")
        if not np.array_equal(np.asarray(saved["scale"]), classifier_scale):
            raise RuntimeError("Classifier scale mismatch")
        model = PlainThreeClassMLP(
            classifier_config["input_dim"],
            classifier_config["width"],
            classifier_config["hidden_layers"],
        ).to(device)
        model.load_state_dict(saved["state_dict"], strict=True)
        model.eval()
        loaded.append({"model": model, "history": saved["history"]})
    print("Loaded", len(loaded), "frozen CE members")
    return loaded

def train_classifier_ensemble():
    train_tensor = torch.as_tensor(train_groups, dtype=torch.float32)
    validation_tensor = torch.as_tensor(validation_groups, dtype=torch.float32)
    trained = []
    for member in range(CLASSIFIER_MEMBERS):
        member_seed = SEED + 20_003 + 10_007 * member
        seed_everything(member_seed)
        model = PlainThreeClassMLP(
            classifier_config["input_dim"],
            classifier_config["width"],
            classifier_config["hidden_layers"],
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LEARNING_RATE)
        loader_generator = torch.Generator().manual_seed(member_seed + 1)
        loader = DataLoader(
            TensorDataset(train_tensor),
            batch_size=max(1, CLASSIFIER_ROW_BATCH_BUDGET // 3),
            shuffle=True,
            generator=loader_generator,
        )
        history = {"train_ce": [], "validation_ce": [], "learning_rate": []}
        best_state, best_value, stale = None, math.inf, 0
        for epoch in range(CLASSIFIER_EPOCHS):
            learning_rate = set_learning_rate(optimizer, epoch)
            model.train()
            running, rows = 0.0, 0
            for (group_batch,) in loader:
                optimizer.zero_grad(set_to_none=True)
                ce = class_ce(model, group_batch)
                if not bool(torch.isfinite(ce)):
                    raise FloatingPointError("Non-finite classifier CE")
                ce.backward()
                optimizer.step()
                running += float(ce.detach().cpu()) * len(group_batch)
                rows += len(group_batch)
            heldout = validation_ce(model, validation_tensor)
            history["train_ce"].append(running / rows)
            history["validation_ce"].append(heldout)
            history["learning_rate"].append(learning_rate)
            if heldout < best_value - 1e-6:
                best_value = heldout
                best_state = copy.deepcopy(model.state_dict())
                stale = 0
            else:
                stale += 1
            if (
                epoch == 0
                or (epoch + 1) % LEARNING_RATE_STEP_EPOCHS == 0
                or epoch + 1 == CLASSIFIER_EPOCHS
            ):
                print(
                    f"member {member + 1}/{CLASSIFIER_MEMBERS}, "
                    f"epoch {epoch + 1:03d}: CE={history['train_ce'][-1]:.5f}, "
                    f"val={heldout:.5f}, lr={learning_rate:.1e}"
                )
            if stale >= CLASSIFIER_EPOCHS:
                break
        if best_state is None:
            raise RuntimeError("Classifier training produced no finite checkpoint")
        model.load_state_dict(best_state)
        model.eval()
        payload = {
            "state_dict": best_state,
            "config": {**classifier_config, "member": member},
            "center": classifier_center,
            "scale": classifier_scale,
            "history": history,
        }
        torch.save(payload, checkpoint_path(member))
        trained.append({"model": model, "history": history})
    return trained

classifier = load_classifier_ensemble()
if classifier is None:
    classifier = train_classifier_ensemble()
print("Classifier device:", device)
print("Parameters/member:", sum(
    parameter.numel() for parameter in classifier[0]["model"].parameters()
))


## 3. Freeze first, diagnose second

The deployed ratio is the arithmetic mean of the **member-wise positive softmax quotients**. We do not average logits, exponentiate a learned log ratio, or tune a checkpoint with the checks below.

Two checks are especially informative:

- \(E_{q_\phi(\theta\mid x)}[r_P(\theta,x)]=1\) and \(E_{q_\eta(x\mid\theta)}[r_L(\theta,x)]=1\);
- the posterior and likelihood routes should satisfy, up to the evidence constant,
  \[
  \log p(\theta)+\log q_\eta(x\mid\theta)+\log r_L
  -\log q_\phi(\theta\mid x)-\log r_P=\log p(x).
  \]

The bridge standard deviation therefore measures *route disagreement*, not posterior accuracy by itself.


In [ ]:
# Frozen CE inference and post-training audits.
@torch.no_grad()
def predict_ratios(points, batch_size=16_384):
    points = np.asarray(points, dtype=np.float32)
    posterior_chunks, likelihood_chunks = [], []
    tiny = torch.finfo(torch.float64).tiny
    for start in range(0, len(points), int(batch_size)):
        stop = min(len(points), start + int(batch_size))
        standardized = transform_classifier_points(points[start:stop])
        tensor = torch.as_tensor(standardized, device=device)
        posterior_mean = None
        likelihood_mean = None
        for pack in classifier:
            probabilities = torch.softmax(
                pack["model"](tensor).to(torch.float64), dim=1
            )
            r_p = (
                probabilities[:, CLASS_S]
                / probabilities[:, CLASS_P].clamp_min(tiny)
            )
            r_l = (
                probabilities[:, CLASS_S]
                / probabilities[:, CLASS_L].clamp_min(tiny)
            )
            scale = float(len(classifier))
            posterior_mean = (
                r_p / scale if posterior_mean is None
                else posterior_mean + r_p / scale
            )
            likelihood_mean = (
                r_l / scale if likelihood_mean is None
                else likelihood_mean + r_l / scale
            )
        posterior_chunks.append(posterior_mean.cpu().numpy())
        likelihood_chunks.append(likelihood_mean.cpu().numpy())
    result = np.column_stack([
        np.concatenate(posterior_chunks),
        np.concatenate(likelihood_chunks),
    ])
    if not np.isfinite(result).all() or np.any(result <= 0.0):
        raise FloatingPointError("Invalid deployed CE probability quotient")
    return result

validation_tensor = torch.as_tensor(validation_groups, dtype=torch.float32)
member_validation_ce = [
    validation_ce(pack["model"], validation_tensor) for pack in classifier
]

audit_p_points = legacy["audit_posterior_points"]
audit_l_points = legacy["audit_likelihood_points"]
audit_p_group = legacy["audit_posterior_group"].astype(int)
audit_l_group = legacy["audit_likelihood_group"].astype(int)
r_p_audit = predict_ratios(audit_p_points)[:, 0]
r_l_audit = predict_ratios(audit_l_points)[:, 1]

def grouped_means(values, group_ids):
    n_groups = int(group_ids.max()) + 1
    sums = np.bincount(group_ids, weights=values, minlength=n_groups)
    counts = np.bincount(group_ids, minlength=n_groups)
    return sums / counts

z_p = grouped_means(r_p_audit, audit_p_group)
z_l = grouped_means(r_l_audit, audit_l_group)

prior_theta = legacy["prior_theta"].astype(np.float32)
observation_rows = np.zeros_like(prior_theta)
prior_points = np.column_stack([prior_theta, observation_rows])
prior_ratios = predict_ratios(prior_points)
log_prior = -np.log(16.0)
pure_bridge = (
    log_prior
    + legacy["joint_log_likelihood"]
    - legacy["joint_log_posterior"]
)
separate_bridge = (
    log_prior
    + legacy["separate_log_likelihood"]
    - legacy["separate_log_posterior"]
)
corrected_bridge = (
    separate_bridge
    + np.log(prior_ratios[:, 1])
    - np.log(prior_ratios[:, 0])
)

diagnostics = pd.DataFrame([
    {
        "quantity": "held-out CE (member mean)",
        "value": float(np.mean(member_validation_ce)),
    },
    {
        "quantity": "RMS log E_qphi[r_P]",
        "value": float(np.sqrt(np.mean(np.log(z_p) ** 2))),
    },
    {
        "quantity": "RMS log E_qeta[r_L]",
        "value": float(np.sqrt(np.mean(np.log(z_l) ** 2))),
    },
    {
        "quantity": "bridge std: original joint JANA",
        "value": float(np.std(pure_bridge)),
    },
    {
        "quantity": "bridge std: matched separate flows",
        "value": float(np.std(separate_bridge)),
    },
    {
        "quantity": "bridge std: matched flows + CE",
        "value": float(np.std(corrected_bridge)),
    },
])
display(diagnostics)


In [ ]:
# Build posterior samples through both routes.
def normalized_positive(values):
    values = np.asarray(values, dtype=np.float64)
    if not np.isfinite(values).all() or np.any(values <= 0.0):
        raise FloatingPointError("Invalid positive weights")
    return values / values.sum(dtype=np.float64)

def normalized_from_log(log_values):
    log_values = np.asarray(log_values, dtype=np.float64)
    if not np.isfinite(log_values).all():
        raise FloatingPointError("Invalid log weights")
    shifted = log_values - np.max(log_values)
    return normalized_positive(np.exp(shifted))

def resample(values, weights, n_samples, seed):
    rng = np.random.default_rng(int(seed))
    index = rng.choice(
        len(values), size=int(n_samples), replace=True, p=weights
    )
    return np.asarray(values[index], dtype=np.float32)

def unweighted_sample(values, n_samples, seed):
    rng = np.random.default_rng(int(seed))
    index = rng.choice(
        len(values),
        size=int(n_samples),
        replace=int(n_samples) > len(values),
    )
    return np.asarray(values[index], dtype=np.float32)

N_EVALUATION = min(10_000, POSTERIOR_SAMPLES)
joint_direct = unweighted_sample(
    legacy["joint_posterior_samples"], N_EVALUATION, SEED + 30_001
)
separate_proposal = legacy["separate_posterior_proposals"].astype(np.float32)
proposal_points = np.column_stack([
    separate_proposal,
    np.zeros_like(separate_proposal),
])
proposal_r_p = predict_ratios(proposal_points)[:, 0]
hybrid_posterior_weights = normalized_positive(proposal_r_p)
separate_direct = unweighted_sample(
    separate_proposal, N_EVALUATION, SEED + 30_002
)
hybrid_posterior = resample(
    separate_proposal,
    hybrid_posterior_weights,
    N_EVALUATION,
    SEED + 30_003,
)

joint_likelihood_weights = normalized_from_log(
    legacy["joint_log_likelihood"]
)
joint_likelihood_route = resample(
    prior_theta,
    joint_likelihood_weights,
    N_EVALUATION,
    SEED + 30_004,
)
hybrid_likelihood_weights = normalized_from_log(
    legacy["separate_log_likelihood"] + np.log(prior_ratios[:, 1])
)
hybrid_likelihood = resample(
    prior_theta,
    hybrid_likelihood_weights,
    N_EVALUATION,
    SEED + 30_005,
)

def ess(weights):
    weights = np.asarray(weights, dtype=np.float64)
    return float(1.0 / np.sum(weights ** 2))

def inside_prior(values):
    values = np.asarray(values)
    return np.all((values >= -2.0) & (values <= 2.0), axis=1)

inference_diagnostics = pd.DataFrame([
    {
        "route": "hybrid posterior proposal",
        "ESS": ess(hybrid_posterior_weights),
        "ESS fraction": (
            ess(hybrid_posterior_weights) / len(hybrid_posterior_weights)
        ),
        "fraction inside prior": float(inside_prior(separate_proposal).mean()),
    },
    {
        "route": "original JANA likelihood",
        "ESS": ess(joint_likelihood_weights),
        "ESS fraction": (
            ess(joint_likelihood_weights) / len(joint_likelihood_weights)
        ),
        "fraction inside prior": 1.0,
    },
    {
        "route": "hybrid likelihood",
        "ESS": ess(hybrid_likelihood_weights),
        "ESS fraction": (
            ess(hybrid_likelihood_weights) / len(hybrid_likelihood_weights)
        ),
        "fraction inside prior": 1.0,
    },
])
display(inference_diagnostics)

result_path = ARTIFACT_ROOT / "posterior_route_samples.npz"
np.savez_compressed(
    result_path,
    analytic=legacy["analytic_reference"],
    jana_joint_posterior=joint_direct,
    matched_separate_posterior=separate_direct,
    hybrid_posterior_ce=hybrid_posterior,
    jana_joint_likelihood_route=joint_likelihood_route,
    hybrid_likelihood_ce=hybrid_likelihood,
)
print("Saved:", result_path)


## 4. Paper-matched posterior metric and visual comparison

For direct comparability, the table below uses the JANA repository's multiscale Gaussian-kernel MMD estimator and exactly 1,000 posterior points per method. This is intentionally distinct from the median-bandwidth MMD used by the broader Exercise 9b campaign.


In [ ]:
# Exact multiscale Gaussian MMD used by JANA-Paper.
from scipy.spatial.distance import cdist

PAPER_SIGMAS = np.asarray([
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 5, 10, 15,
    20, 25, 30, 35, 100, 1e3, 1e4, 1e5, 1e6,
], dtype=np.float64)

def paper_kernel(distance2):
    result = np.zeros_like(distance2, dtype=np.float64)
    for sigma in PAPER_SIGMAS:
        result += np.exp(-distance2 / (2.0 * sigma))
    return result

def paper_mmd(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)
    k_xx = paper_kernel(
        cdist(left, left, metric="sqeuclidean")
    ).mean()
    k_yy = paper_kernel(
        cdist(right, right, metric="sqeuclidean")
    ).mean()
    k_xy = paper_kernel(
        cdist(left, right, metric="sqeuclidean")
    ).mean()
    return float(np.sqrt(max(0.0, k_xx + k_yy - 2.0 * k_xy)))

metric_rng = np.random.default_rng(SEED + 40_001)
reference = legacy["analytic_reference"]
reference_1000 = reference[
    metric_rng.choice(
        len(reference), size=1_000, replace=len(reference) < 1_000
    )
]
methods = {
    "Original JANA q_phi": joint_direct,
    "Matched separate q_phi": separate_direct,
    "Hybrid q_phi + CE": hybrid_posterior,
    "Original JANA q_eta route": joint_likelihood_route,
    "Hybrid q_eta + CE": hybrid_likelihood,
}
metric_rows = []
plot_samples = {"Analytical": reference_1000}
for name, values in methods.items():
    index = metric_rng.choice(
        len(values), size=1_000, replace=len(values) < 1_000
    )
    selected = values[index]
    plot_samples[name] = selected
    metric_rows.append({
        "method": name,
        "paper multiscale MMD": paper_mmd(reference_1000, selected),
    })
metric_table = pd.DataFrame(metric_rows).sort_values(
    "paper multiscale MMD"
)
display(metric_table)
metric_table.to_csv(
    ARTIFACT_ROOT / "paper_multiscale_mmd.csv", index=False
)


In [ ]:
# Main comparison figure.
import matplotlib.pyplot as plt

figure, axes = plt.subplots(
    2, 3, figsize=(12, 8), sharex=True, sharey=True
)
titles = [
    "Analytical",
    "Original JANA q_phi\n(paper training)",
    "Matched separate q_phi\n(before CE)",
    "Hybrid q_phi × CE",
    "Original JANA\nprior × q_eta",
    "Hybrid\nprior × q_eta × CE",
]
keys = [
    "Analytical",
    "Original JANA q_phi",
    "Matched separate q_phi",
    "Hybrid q_phi + CE",
    "Original JANA q_eta route",
    "Hybrid q_eta + CE",
]
for axis, title, key in zip(axes.flat, titles, keys):
    values = plot_samples[key]
    axis.scatter(
        values[:, 0],
        values[:, 1],
        s=5,
        alpha=0.45,
        color="#4cc9f0",
        linewidths=0,
    )
    axis.set_title(title, fontsize=11)
    axis.set_xlim(-2, 2)
    axis.set_ylim(-2, 2)
    axis.set_aspect("equal")
    axis.set_facecolor("#002040")
    axis.grid(False)
    axis.tick_params(colors="#536777", labelsize=8)
    for spine in axis.spines.values():
        spine.set_visible(False)
for axis in axes[:, 0]:
    axis.set_ylabel(r"$\theta_2$")
for axis in axes[-1, :]:
    axis.set_xlabel(r"$\theta_1$")
figure.suptitle(
    "Wiqvist Two Moons at x = (0, 0): exact JANA topology versus hybrid CE",
    fontsize=14,
)
figure.tight_layout()
figure_path = (
    ARTIFACT_ROOT / "jana_twomoons_exact_vs_hybrid_ce.png"
)
figure.savefig(figure_path, dpi=220, bbox_inches="tight")
figure.savefig(
    ARTIFACT_ROOT / "jana_twomoons_exact_vs_hybrid_ce.pdf",
    bbox_inches="tight",
)
plt.show()
print("Saved:", figure_path)


In [ ]:
# Training histories: legacy NLLs and the ten CE members.
history_paths = {
    "joint JANA": ARTIFACT_ROOT / "joint_jana_history.csv",
    "separate posterior": (
        ARTIFACT_ROOT / "separate_posterior_history.csv"
    ),
    "separate likelihood": (
        ARTIFACT_ROOT / "separate_likelihood_history.csv"
    ),
}
for label, path in history_paths.items():
    frame = pd.read_csv(path)
    print(
        label,
        "columns:",
        list(frame.columns),
        "rows:",
        len(frame),
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, path in history_paths.items():
    frame = pd.read_csv(path)
    candidates = [
        column for column in frame.columns
        if "loss" in column.lower() and "val" not in column.lower()
    ]
    for column in candidates:
        axes[0].plot(
            frame[column].to_numpy(),
            alpha=0.8,
            label=f"{label}: {column}",
        )
for member, pack in enumerate(classifier):
    axes[1].plot(
        pack["history"]["validation_ce"],
        alpha=0.55,
        label=f"member {member + 1}",
    )
axes[0].set(
    title="Legacy BayesFlow training",
    xlabel="stored step",
    ylabel="NLL",
)
axes[1].set(
    title="Frozen-checkpoint selection",
    xlabel="epoch",
    ylabel="held-out CE",
)
for axis in axes:
    axis.grid(alpha=0.2)
axes[0].legend(fontsize=7)
if len(classifier) <= 4:
    axes[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    ARTIFACT_ROOT / "training_histories.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()


## 5. How to interpret the run

This experiment separates three possible explanations for the earlier discrepancy.

- If **original JANA now recovers both crescents**, the poor result in `Exercise_9b_SBIBM_TwoMoons.ipynb` was not evidence against the published JANA result. It arose from changing the simulator and replacing the historical architecture/training campaign.
- If original JANA works but the **matched separate flows** do not, the remaining issue is optimization variability or the separation of the two optimizers, not flow capacity.
- If the matched separate flows are imperfect but **CE improves both posterior and likelihood routes**, the classifier is doing the intended residual correction.
- If CE improves one route but worsens the other, inspect the two normalization audits and the corrected bridge variance. That pattern identifies which conditional reference or density ratio is failing.
- Agreement between the corrected posterior and likelihood routes is an internal consistency test, not an independent truth test. The analytical posterior and paper-matched MMD remain the external benchmark.

For a literal ten-repetition reproduction of the paper's reliability study, rerun the notebook with `SEED = 1, …, 10` into separate artifact directories. The default run is seed 1, the repetition displayed in the main paper.


In [ ]:
# Final immutable run manifest.
manifest = {
    "status": "complete",
    "profile": PROFILE,
    "seed": SEED,
    "simulation_budget": SIMULATION_BUDGET,
    "legacy_metadata": metadata,
    "classifier_config": classifier_config,
    "diagnostics": diagnostics.to_dict(orient="records"),
    "inference_diagnostics": (
        inference_diagnostics.to_dict(orient="records")
    ),
    "paper_multiscale_mmd": metric_table.to_dict(orient="records"),
    "result_path": str(result_path),
    "figure_path": str(figure_path),
}
manifest_path = ARTIFACT_ROOT / "run_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True)
)
print("Run complete:", manifest_path)
